In [0]:
# SCD Type 1: upsert source into target Delta table (latest record wins, no history kept)
from delta.tables import DeltaTable
from pyspark.sql import functions as F

source_table = "main.default.customers_source"   # incoming data (e.g. latest daily extract)
target_table = "main.default.customers_target"     # SCD1 Delta table
join_keys = ["customer_id"]                        # business / natural key(s)

source_df = spark.table(source_table).dropDuplicates(subset=join_keys)  # keep one row per key

# Column comparison: update only when any tracked attribute actually changed
tracked_cols = [c for c in source_df.columns if c not in join_keys]
update_condition = " OR ".join(
    [f"target.{c} IS DISTINCT FROM source.{c}" for c in tracked_cols]
)

# (One-time) create the target table if it doesn't exist yet
if not spark.catalog.tableExists(target_table):
    source_df.write.format("delta").saveAsTable(target_table)
else:
    target_dt = DeltaTable.forName(spark, target_table)

    target_dt.alias("target").merge(
        source_df.alias("source"),
        " AND ".join([f"target.{k} = source.{k}" for k in join_keys]),
    ).whenMatchedUpdate(
        condition=update_condition,
        set={c: f"source.{c}" for c in tracked_cols},
    ).whenNotMatchedInsert(
        values={c: f"source.{c}" for c in source_df.columns},
    ).execute()

print(f"SCD1 merge completed into {target_table}")
spark.table(target_table).display()

In [0]:
# SCD Type 2: keep full change history in the target Delta table
# Each change CLOSES the old row (end_ts set, is_current = false) and
# INSERTS a new current row (end_ts = null, is_current = true).
#
# Trick: a single MERGE cannot UPDATE and INSERT the same key in one pass,
# so each changed key is staged TWICE:
#   - one row with merge_key = <key>     -> matches the old current row -> closes it
#   - one row with merge_key = NULL      -> matches nothing            -> inserts the new version
# Brand-new keys are staged once with merge_key = <key>; no current row exists,
# so they fall through to whenNotMatchedInsert as well.
from delta.tables import DeltaTable
from pyspark.sql import functions as F

source_table = "main.default.customers_source"      # incoming data (e.g. latest daily extract)
target_table = "main.default.customers_target_scd2" # SCD2 Delta table
join_keys = ["customer_id"]                          # business / natural key(s)

start_ts_col, end_ts_col, current_col = "start_ts", "end_ts", "is_current"
key_col = join_keys[0]  # this pattern assumes a single business key

source_df = spark.table(source_table).dropDuplicates(subset=join_keys)
key_dtype = source_df.schema[key_col].dataType
tracked_cols = [c for c in source_df.columns if c not in join_keys]
changed_condition = " OR ".join(
    [f"target.{c} IS DISTINCT FROM source.{c}" for c in tracked_cols]
)

if not spark.catalog.tableExists(target_table):
    # One-time initial load: every source row is a brand-new current version
    (source_df
     .withColumn(start_ts_col, F.current_timestamp())
     .withColumn(end_ts_col, F.lit(None).cast("timestamp"))
     .withColumn(current_col, F.lit(True))
     .write.format("delta").saveAsTable(target_table))
else:
    target_dt = DeltaTable.forName(spark, target_table)
    current_target = (spark.table(target_table)
                      .where(F.col(current_col))
                      .alias("target"))

    # Classify source rows: changed keys vs brand-new keys (unchanged keys are skipped)
    changed_rows = (
        source_df.alias("source")
        .join(current_target, join_keys)
        .where(changed_condition)
        .select("source.*")
    )
    new_rows = (
        source_df.alias("source")
        .join(current_target, join_keys, "left_anti")
        .select("source.*")
    )

    src_cols = source_df.columns
    staged = (
        changed_rows
        .select(F.lit(None).cast(key_dtype).alias("merge_key"), *src_cols)  # insert new version
        .unionByName(changed_rows
        .select(F.col(key_col).cast(key_dtype).alias("merge_key"), *src_cols))  # close old row
        .unionByName(new_rows
        .select(F.col(key_col).cast(key_dtype).alias("merge_key"), *src_cols))  # brand-new key
    )

    (target_dt.alias("target").merge(
        staged.alias("staged"),
        f"target.{key_col} = staged.merge_key AND target.{current_col} = true",
    )
     .whenMatchedUpdate(set={  # close the old version
         end_ts_col: F.current_timestamp(),
         current_col: F.lit(False),
     })
     .whenNotMatchedInsert(values={  # insert the new/current version
         **{c: F.col(f"staged.{c}") for c in src_cols},
         start_ts_col: F.current_timestamp(),
         end_ts_col: F.lit(None).cast("timestamp"),
         current_col: F.lit(True),
     })
     .execute())

print(f"SCD2 merge completed into {target_table}")
spark.table(target_table).orderBy(key_col, start_ts_col).display()

In [0]:
# SCD2 sanity check: current snapshot should match the source (one row per key),
# and the full history should contain one closed row per past change.
full = spark.table(target_table)

print("Rows total / current:", full.count(), "/", full.where(F.col(current_col)).count())

full.select(key_col, *tracked_cols, start_ts_col, end_ts_col, current_col) \
    .orderBy(key_col, F.col(current_col).desc(), start_ts_col) \
    .display()